# セミナーII 第2回 実践：写真1枚から奥行きを推定する（Depth Anything V2）

写真1枚から、写っている物それぞれがどれくらい近いか（奥行き）を推定するモデル **Depth Anything V2** を動かします。講義で見た ViT（Vision Transformer）が中で使われています。

難しいことは考えずに、上のセルから順に実行してください。コードを読めなくてもかまいません。何ができて、何が苦手かを、自分の目で確かめることを目標にします。

**最初に**

1. メニューの「ファイル」→「ドライブにコピーを保存」で、自分のコピーを作ります。
2. 「ランタイム」→「ランタイムのタイプを変更」で **GPU**（T4 など）を選びます。
3. 上のセルから順に実行します。セルの左の ▶ を押すか、セルを選んで **Shift + Enter** を押すと実行できます。

ログインや Google ドライブへのアクセスの許可は必要ありません。全体で約45分（うち計算 約5分）です。各節の見出しの下に目安を書いています。

**設定の選び方**：セルの右に入力欄やプルダウンが出るセルでは、そこで値を選んでから実行します（コードの `= ` の後ろを書き換えても同じです）。**関数を定義するセル**は、コードを畳んであります。実行するだけでかまいません（ダブルクリックで中身を見られます）。

**うまくいかないとき**：しばらく操作しないと接続が切れ、読み込んだモデルや画像が消えます。`NameError` が出たときなどは、1. から順に実行し直してください。

## 0. このノートブックの流れ

目安：約1分

| 節 | やること |
| --- | --- |
| 1. 準備 | ライブラリとモデルを読み込む |
| 2. 手法を知る | 写真1枚から奥行きを推定するとは何か。Depth Anything V2 の仕組みと、学習のしかた |
| 3. 例の画像で試す | 用意した4枚の写真で推定する |
| 4. 自分の画像で試す | 自分で撮った写真を入れる |
| 5. やってみよう | 入力の大きさ、モデルの大きさを変える。メートル単位で推定して、正解と比べる |
| 6. 活用 | 奥行きを使って、背景をぼかす、ピントを動かす、霧をかける、3次元で見る |
| 7. まとめ | 分かったことと、苦手なこと |

## 1. 準備

目安：約3分（うち計算 約1分）

使うライブラリを読み込み、計算に使う装置（`device`）を決めます。`device: cuda` と GPU の名前が表示されれば準備完了です。`device: cpu` と表示されたら、「ランタイム」→「ランタイムのタイプを変更」で GPU を選んで、もう一度実行してください。

In [ ]:
import importlib.util
import io
import subprocess
import sys
import time

# Colab に入っていないライブラリがあれば入れる
for package in ["transformers", "plotly", "pooch"]:
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Image as ShowImage, display
from PIL import Image
from skimage import data
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32  # GPU では半分の精度で速く計算する
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "（GPU を選ぶと速くなります）")

このノートブックで使う関数を定義します。モデルの読み込み、奥行きの推定、図の表示などです。

In [ ]:
# @title 関数を定義する（実行するだけでよい。ダブルクリックでコードを表示） { display-mode: "form" }
MODEL_SMALL = "depth-anything/Depth-Anything-V2-Small-hf"  # 相対的な奥行き。約2500万パラメータ
METRIC_MODEL = "depth-anything/Depth-Anything-V2-Metric-Indoor-Small-hf"  # メートル単位（屋内用）。5-3 で変えられる
_loaded = {}


def load_model(model_id):
    # モデル（と、その前処理）を読み込む。2回目からは、読み込んだものを使い回す
    if model_id not in _loaded:
        processor = AutoImageProcessor.from_pretrained(model_id)
        model = AutoModelForDepthEstimation.from_pretrained(model_id, dtype=dtype).to(device).eval()
        _loaded[model_id] = (processor, model)
    return _loaded[model_id]


def estimate_depth(image, model_id=MODEL_SMALL, input_size=None, return_info=False):
    # 画像1枚の奥行きを推定し、画像と同じ縦×横の配列で返す
    processor, model = load_model(model_id)
    size = {}  # 指定しなければ、モデルの標準（短い辺を 518 程度）
    if input_size is not None:  # 短い辺を input_size にし、縦横を 14 の倍数にそろえる
        scale = input_size / min(image.size)
        size = {"size": {"height": max(14, round(image.height * scale / 14) * 14),
                         "width": max(14, round(image.width * scale / 14) * 14)}, "keep_aspect_ratio": False}
    pixel_values = processor(images=image, return_tensors="pt", **size)["pixel_values"].to(device, dtype)
    if device == "cuda":
        torch.cuda.synchronize()
    start = time.time()
    with torch.no_grad():
        outputs = model(pixel_values=pixel_values)
    if device == "cuda":
        torch.cuda.synchronize()
    seconds = time.time() - start
    depth = processor.post_process_depth_estimation(outputs, target_sizes=[(image.height, image.width)])
    depth = depth[0]["predicted_depth"].float().cpu().numpy()
    if not return_info:
        return depth
    h, w = pixel_values.shape[-2:]
    patch = model.config.backbone_config.patch_size
    return depth, {"input": (h, w), "tokens": (h // patch) * (w // patch), "seconds": seconds}


def count_params(model_id):
    return sum(p.numel() for p in load_model(model_id)[1].parameters())


def normalize(depth):
    # 0（遠い）〜1（近い）にそろえる。飛び抜けた値に引きずられないよう、上下1%を除いた範囲を使う
    low, high = np.percentile(depth, [1, 99])
    return np.clip((depth - low) / (high - low + 1e-8), 0, 1)


def fit_size(image, max_side=1024):
    # 大きすぎる写真は縮小する（計算と表示を速くするため）
    image = image.convert("RGB")
    scale = max_side / max(image.size)
    if scale < 1:
        image = image.resize((round(image.width * scale), round(image.height * scale)), Image.LANCZOS)
    return image


def show_depth(image, depth, title="", cmap="inferno", label="near ← → far"):
    # 入力の画像と、推定した奥行き（明るいほど近い）を並べる
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    axes[0].imshow(image)
    axes[0].set_title(title)
    shown = axes[1].imshow(depth, cmap=cmap)
    axes[1].set_title("depth (bright = near)")
    for ax in axes:
        ax.axis("off")
    fig.colorbar(shown, ax=axes[1], fraction=0.035)
    plt.tight_layout()
    plt.show()


def disk(radius):
    # 円形のぼかし（レンズのボケに近い形）の重み
    r = int(np.ceil(radius))
    y, x = np.mgrid[-r:r + 1, -r:r + 1]
    kernel = (x ** 2 + y ** 2 <= radius ** 2).astype(np.float32)
    return kernel / kernel.sum()


def blur_stack(image, max_radius, levels=8):
    # ぼかしの半径を変えた画像を、まとめて作っておく
    img = np.asarray(image, dtype=np.float32)
    radii = np.linspace(0, max_radius, levels)
    return radii, np.stack([img if r < 0.5 else cv2.filter2D(img, -1, disk(r)) for r in radii])


def compose_blur(radii, stack, radius):
    # 画素ごとに、ぼかしの半径 radius に近い2枚を混ぜる
    pos = np.interp(radius, radii, np.arange(len(radii)))
    low = np.floor(pos).astype(int)
    high = np.minimum(low + 1, len(radii) - 1)
    t = (pos - low)[..., None]
    rows, cols = np.indices(radius.shape)
    out = stack[low, rows, cols] * (1 - t) + stack[high, rows, cols] * t
    return np.clip(out, 0, 255).astype(np.uint8)


def depth_of_field(image, depth, focus, strength, levels=8, stack=None):
    # ピントを合わせる近さ focus（0〜1）から離れた画素ほど、大きくぼかす
    near = normalize(depth)
    radius = np.abs(near - focus) * strength
    radii, stack = stack if stack is not None else blur_stack(image, strength, levels)
    return compose_blur(radii, stack, radius)


def show_gif(frames, duration=120):
    # 画像の列を、くり返し再生する GIF にして表示する
    buffer = io.BytesIO()
    frames = [Image.fromarray(f) for f in frames]
    frames[0].save(buffer, format="GIF", save_all=True, append_images=frames[1:], duration=duration, loop=0)
    display(ShowImage(data=buffer.getvalue(), format="gif"))


print("関数を定義しました")

Depth Anything V2 の Small（約2500万パラメータ）を読み込みます。初回は重み（約100MB）をダウンロードします。`HF_TOKEN` に関する警告が表示されても無視してかまいません。

- モデル：[depth-anything/Depth-Anything-V2-Small-hf](https://huggingface.co/depth-anything/Depth-Anything-V2-Small-hf)（Apache-2.0）
- 論文：L. Yang et al., "Depth Anything V2," NeurIPS 2024（[arXiv:2406.09414](https://arxiv.org/abs/2406.09414)）

In [ ]:
processor, model = load_model(MODEL_SMALL)
print(f"パラメータの数: {count_params(MODEL_SMALL) / 1e6:.1f}M")

## 2. 手法を知る

目安：約10分

### 2-1. 写真1枚から奥行きを推定するとは

**単眼深度推定**（monocular depth estimation）は、カメラ1台で撮った写真1枚から、画素ごとに奥行きを推定する問題です。

第1回の画像分類は、画像1枚に答えが1つ（37クラスのどれか）でした。奥行きの推定では、**画素の数だけ答え**があります。式で書けば同じ $Y = F(X; \theta)$ ですが、$Y$ が画像と同じ縦×横の配列になります。このように画素ごとに答えを出す問題を、**密な予測**（dense prediction）と呼びます。

写真は3次元の場面を2次元に写したものなので、奥行きの情報は写真から消えています。近くの小さな物と、遠くの大きな物は、同じ写真になることがあります。それでも人間は、片目でも、次のような手がかりから奥行きを感じ取れます。

| 手がかり | 例 |
| --- | --- |
| 大きさ | 同じ種類の物なら、小さく写っているほど遠い |
| 重なり | 手前の物が、奥の物を隠す |
| 遠近法 | 平行な線（道路、線路）が、遠くで1点に集まる |
| きめ | 地面の模様は、遠いほど細かく見える |
| かすみ | 遠くの景色ほど、白っぽくかすむ |

奥行きを測るほかの方法には、2台のカメラの見え方のずれから三角測量で求める**ステレオカメラ**や、光を当てて返ってくるまでの時間を測る **LiDAR**、**ToF センサ**があります。単眼深度推定のモデルは、こうした手がかりを、大量の画像から学習で身につけます。

### 2-2. 相対的な奥行きと、メートル単位の奥行き

Depth Anything V2 の基本のモデルが出すのは、**相対的な奥行き**です。値が大きいほど近いことを表しますが、単位はありません。「どちらが近いか」「どこで奥行きが変わるか」は分かっても、「何メートルか」は分かりません（論文の言葉では、affine-invariant inverse depth：距離の逆数に近い量で、定数倍とずれの分は決まらない）。

メートル単位の奥行きが必要なときは、メートル単位の正解が付いたデータで追加学習（ファインチューニング）したモデルを使います。Depth Anything V2 には、屋内用と屋外用があります。5-3 で使います。

### 2-3. Depth Anything V2 の仕組み

| 段階 | 処理 |
| --- | --- |
| ① 前処理 | 写真の短い辺を 518 画素程度にそろえ、縦横を 14 の倍数にする |
| ② パッチに分ける | 14×14 画素の正方形（パッチ）に分け、1つのパッチを1つのトークン（ベクトル）にする |
| ③ エンコーダ（ViT） | トークンの間でアテンションを計算する Transformer ブロックを12回重ねる。重みは **DINOv2**（大量の画像から、正解のラベルなしで学習した ViT）から始めている |
| ④ デコーダ（DPT） | 途中の4つの層（3、6、9、12番目）のトークンを、もとの画像の形に並べ直して4枚の特徴マップにする。それらを合わせながら細かくしていき、画素ごとの奥行きを出す |

講義の ViT は 16×16 画素のパッチでしたが、DINOv2 は 14×14 画素です。518 × 518 の入力なら、縦横に 518 ÷ 14 = 37 個ずつ、37 × 37 = **1369 個のトークン**になります。アテンションでは、どのトークンも1層目から画像全体のトークンと関係を計算できるので、写真全体の手がかり（遠近法の線など）を使えます。

④ のデコーダは DPT（Dense Prediction Transformer）と呼ばれる仕組みです（R. Ranftl et al., "Vision Transformers for Dense Prediction," ICCV 2021）。分類では最後に1つの答えにまとめましたが、密な予測では、トークンを画像の形に戻して、画素ごとの答えに広げます。

下のセルで、宇宙飛行士の写真がパッチに分けられる様子を見ます。

In [ ]:
# 例の画像（宇宙飛行士 Eileen Collins の写真。NASA、パブリックドメイン）
astronaut = Image.fromarray(data.astronaut())

pixel_values = processor(images=astronaut, return_tensors="pt")["pixel_values"][0]  # ① 前処理の後
mean, std = torch.tensor(processor.image_mean)[:, None, None], torch.tensor(processor.image_std)[:, None, None]
shown = (pixel_values * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()  # 表示できるように戻す

patch = model.config.backbone_config.patch_size
h, w = shown.shape[:2]
print(f"元の大きさ: {astronaut.height}×{astronaut.width} → 前処理の後: {h}×{w}")
print(f"パッチ {patch}×{patch} 画素 → 縦 {h // patch} × 横 {w // patch} = {(h // patch) * (w // patch)} 個のトークン")
print("デコーダが使う層（番号）:", model.config.backbone_config.out_indices)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, (y0, x0, size) in zip(axes, [(0, 0, h), (100, 180, 140)]):  # 全体と、顔のあたりの拡大
    ax.imshow(shown[y0:y0 + size, x0:x0 + size], extent=(x0, x0 + size, y0 + size, y0))
    for k in range(0, max(h, w) + 1, patch):  # パッチの境目
        ax.axhline(k, color="white", lw=0.4)
        ax.axvline(k, color="white", lw=0.4)
    ax.set_xlim(x0, x0 + size)
    ax.set_ylim(y0 + size, y0)
    ax.axis("off")
axes[0].set_title(f"{h}x{w} input, {patch}x{patch} patches")
axes[1].set_title("zoom: each square = 1 token")
plt.show()

### 2-4. どうやって学習したか

学習の基本は第1回と同じで、予測と正解のずれ（Loss）が小さくなるように $\theta$ を動かします。Depth Anything V2 の工夫は、**どんなデータで学習するか**にあります（論文の Section 3）。

実写の写真にセンサで測った奥行きを付けたデータには、ガラスなど透明な物の奥行きが測れない、細かいところが粗い、といった問題があります。一方、CG で作った合成画像なら、奥行きの正解を画素ごとに正確に作れますが、実写とは見た目が違い、場面の種類も限られます。そこで、次の3段階で学習しています。

| 段階 | 内容 |
| --- | --- |
| ① 先生役のモデル | 大きなモデル（DINOv2 の Giant、約13億パラメータ）を、正確な正解が付いた**合成画像だけ**（5種類のデータセット、約60万枚）で学習する |
| ② 正解を付ける | 先生役のモデルで、正解の無い**実写の写真**（8種類のデータセット、約6200万枚）の奥行きを推定し、仮の正解（疑似ラベル）にする |
| ③ 生徒役のモデル | 配布するモデル（Small、Base、Large）を、仮の正解が付いた実写の写真で学習する |

相対的な奥行きを学習するので、Loss には、予測と正解の**定数倍とずれの違いを無視して比べる** Loss（scale- and shift-invariant loss）を使っています。奥行きの変わり目（物の輪郭）がくっきりするように、奥行きの傾き（勾配）のずれを比べる Loss も加えています。

## 3. 例の画像で試す

目安：約3分

4枚の写真で奥行きを推定します。右の図は推定した奥行きで、**明るいほど近い**ことを表します（相対的な値なので、右の目盛りの数字に単位はありません）。

| 名前 | 写真 | 出典 |
| --- | --- | --- |
| `astronaut` | 宇宙飛行士の肖像 | NASA（パブリックドメイン） |
| `motorcycle` | 室内のバイク | Middlebury 2014 ステレオデータセット（D. Scharstein et al., GCPR 2014）。正解の奥行きが付いている（5-3 で使う） |
| `rocket` | 打ち上げ台のロケット | SpaceX（パブリックドメイン） |
| `coffee` | テーブルの上のコーヒー | 撮影 Rachel Michetti（CC0） |

4枚とも、scikit-image に入っている写真です。

In [ ]:
motorcycle, _, motorcycle_disparity = data.stereo_motorcycle()  # 左のカメラの写真と、正解の視差
images = {
    "astronaut": astronaut,
    "motorcycle": Image.fromarray(motorcycle),
    "rocket": Image.fromarray(data.rocket()),
    "coffee": Image.fromarray(data.coffee()),
}

depths = {}
for name, image in images.items():
    depths[name] = estimate_depth(image)
    show_depth(image, depths[name], name)

**見どころ**

- `astronaut`：抱えたヘルメットが飛び抜けて近く、体と顔はその奥、背景の旗やスペースシャトルの模型はさらに遠いと推定されています。
- `motorcycle`：手前のタイヤから奥の壁まで、なめらかに奥行きが変わっています。細い部品の輪郭も出ています。
- `rocket`：左右の鉄塔が手前、ロケットはその奥で、空は一番遠い（暗い）と推定されています。空のように限りなく遠い所は、相対的な奥行き（距離の逆数に近い量）では0に近い値になります。

## 4. 自分の画像で試す

目安：約5分（画像を用意する時間は除く）

`USE_UPLOAD` にチェックを入れて実行すると、アップロードのボタンが表示されます（Colab のみ）。何枚でも選べます。アップロードした画像は、ファイル名（拡張子を除く）の名前で、5. と 6. でも使えます。

試してほしい写真：遠くまで続く道や風景、物がたくさんある室内、鏡やガラス、壁に貼ったポスターや画面を撮った写真（本当は平らです）。

In [ ]:
USE_UPLOAD = False  # @param {type:"boolean"}

if USE_UPLOAD:
    from google.colab import files

    for filename, content in files.upload().items():
        name = filename.rsplit(".", 1)[0]
        images[name] = fit_size(Image.open(io.BytesIO(content)))
        depths[name] = estimate_depth(images[name])
        show_depth(images[name], depths[name], name)
print("使える画像の名前:", list(images))

## 5. やってみよう：推論の設定を変える

目安：約10分（うち計算 約2分）

学習はしません。推論のときに選べる設定を変えて、結果と時間を比べます。

### 5-1. 入力の大きさを変える

モデルに入れる前にそろえる大きさ（短い辺の画素数。標準は 518）を変えます。大きくすると細かいところまで出ますが、トークンの数が増えます。トークンの数は入力の大きさの2乗に比例し、アテンションの計算はトークンの数の2乗に比例して増えます。

`IMAGE` に画像の名前を選んで（自分の画像は名前を入力して）実行します。

In [ ]:
IMAGE = "motorcycle"  # @param ["astronaut", "motorcycle", "rocket", "coffee"] {allow-input: true}

image = images[IMAGE]
sizes = [266, 518, 770, 1022]  # 短い辺の画素数。どれも 14 の倍数
fig, axes = plt.subplots(1, len(sizes), figsize=(4.5 * len(sizes), 4))
for ax, size in zip(axes, sizes):
    estimate_depth(image, input_size=size)  # 大きさごとに、1回目は準備に時間がかかるので、測る前に1回動かす
    depth, info = estimate_depth(image, input_size=size, return_info=True)
    ax.imshow(depth, cmap="inferno")
    ax.set_title(f"input {info['input'][0]}x{info['input'][1]}\n{info['tokens']} tokens, {info['seconds'] * 1000:.0f} ms")
    ax.axis("off")
    print(f"入力 {size:4d} → {info['input'][0]}×{info['input'][1]}、トークン {info['tokens']:5d} 個、{info['seconds'] * 1000:6.1f} ms")
plt.tight_layout()
plt.show()

細かさと時間は、どう変わったでしょうか。学習は 518 の大きさで行われています。それより大きくしたときに、細かさと全体のまとまりがどう変わるか、自分の画像でも確かめてみましょう。

### 5-2. モデルの大きさを変える

Small のほかに、Base（約1億パラメータ）と Large（約3億パラメータ）があります。Small と比べてみましょう。初回は重みをダウンロードします（Base 約400MB、Large 約1.3GB）。

Base と Large は、**CC BY-NC 4.0**（非商用に限る）のライセンスです。授業や研究で試すのはかまいませんが、商用の製品には使えません。

In [ ]:
IMAGE = "motorcycle"  # @param ["astronaut", "motorcycle", "rocket", "coffee"] {allow-input: true}
MODEL = "Large"  # @param ["Base", "Large"]

image = images[IMAGE]
model_ids = {"Small": MODEL_SMALL, "Base": "depth-anything/Depth-Anything-V2-Base-hf",
             "Large": "depth-anything/Depth-Anything-V2-Large-hf"}
estimate_depth(image, model_ids[MODEL])  # 読み込みと準備（測る前に1回動かしておく）

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
axes[0].imshow(image)
axes[0].set_title(IMAGE)
for ax, size in zip(axes[1:], ["Small", MODEL]):
    depth, info = estimate_depth(image, model_ids[size], return_info=True)
    ax.imshow(depth, cmap="inferno")
    ax.set_title(f"{size}: {count_params(model_ids[size]) / 1e6:.0f}M params, {info['seconds'] * 1000:.0f} ms")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

### 5-3. メートル単位で推定して、正解と比べる

メートル単位の正解が付いた合成画像で追加学習したモデルを使います。屋内用（Hypersim で学習。最大 20 m）と、屋外用（Virtual KITTI 2 で学習。最大 80 m）があります。

`motorcycle` の写真には、ステレオカメラで測った正解の視差（左右のカメラでのずれ）と、カメラの値（焦点距離、カメラの間隔）が付いています。ここから、正解の奥行き（メートル）を三角測量で計算できます。

$$\text{奥行き} = \frac{\text{焦点距離} \times \text{カメラの間隔}}{\text{視差}}$$

まず、屋内用のモデルで `motorcycle` の奥行きを推定して、正解と比べます。

In [ ]:
METRIC = "屋内（Indoor）"  # @param ["屋内（Indoor）", "屋外（Outdoor）"]

metric_ids = {"屋内（Indoor）": "depth-anything/Depth-Anything-V2-Metric-Indoor-Small-hf",
              "屋外（Outdoor）": "depth-anything/Depth-Anything-V2-Metric-Outdoor-Small-hf"}
METRIC_MODEL = metric_ids[METRIC]

# 正解の奥行き。焦点距離 994.978 画素、カメラの間隔 0.193 m、視差の補正 31.086 画素（scikit-image の説明による）
with np.errstate(divide="ignore", invalid="ignore"):
    true_m = 994.978 * 0.193001 / (motorcycle_disparity + 31.086)
valid = np.isfinite(motorcycle_disparity)  # 正解が無い画素（隠れていた所など）を除く

pred_m = estimate_depth(images["motorcycle"], METRIC_MODEL)
error = np.abs(pred_m - true_m)
print(f"正解の奥行き: {true_m[valid].min():.2f}〜{true_m[valid].max():.2f} m")
print(f"推定の奥行き: {pred_m.min():.2f}〜{pred_m.max():.2f} m")
print(f"誤差の中央値: {np.median(error[valid]):.2f} m（正解に対して {np.median(error[valid] / true_m[valid]):.0%}）")

vmin, vmax = np.percentile(true_m[valid], [1, 99])
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (img, title, cmap) in zip(axes, [(np.where(valid, true_m, np.nan), "ground truth (stereo) [m]", "turbo"),
                                         (pred_m, f"predicted ({METRIC_MODEL.split('-')[-3]}) [m]", "turbo"),
                                         (np.where(valid, error, np.nan), "absolute error [m]", "magma")]):
    shown = ax.imshow(img, cmap=cmap, vmin=0 if "error" in title else vmin, vmax=1 if "error" in title else vmax)
    ax.set_title(title)
    ax.axis("off")
    fig.colorbar(shown, ax=ax, fraction=0.035)
plt.tight_layout()
plt.show()

ここでは色が、赤いほど遠く、青いほど近いことを表します（目盛りはメートル）。正解の白い所は、片方のカメラにしか写っていないなどの理由で、正解が無い画素です。屋外用（Outdoor）に変えて実行すると、誤差はどう変わるでしょうか。

次のセルでは、好きな画像の好きな位置までの距離を表示します。`X` と `Y` は、画像の左上を 0、右下を 1 とした位置です。

In [ ]:
IMAGE = "motorcycle"  # @param ["astronaut", "motorcycle", "rocket", "coffee"] {allow-input: true}
X = 0.5  # @param {type:"slider", min:0, max:1, step:0.01}
Y = 0.6  # @param {type:"slider", min:0, max:1, step:0.01}

image = images[IMAGE]
meters = estimate_depth(image, METRIC_MODEL)
col, row = round(X * (image.width - 1)), round(Y * (image.height - 1))
print(f"{IMAGE} の ({X:.2f}, {Y:.2f}) までの推定の距離: {meters[row, col]:.2f} m（{METRIC}）")

plt.figure(figsize=(6, 5))
plt.imshow(image)
plt.plot(col, row, "+", color="red", markersize=25, markeredgewidth=3)
plt.title(f"{meters[row, col]:.2f} m")
plt.axis("off")
plt.show()

## 6. 活用：奥行きを使う

目安：約12分（うち計算 約1分）

推定した奥行きを、別の処理につなげます。研究でも、モデルの出力をそのまま使うより、ほかの処理の材料にすることがよくあります。

### 6-1. 背景をぼかす（被写界深度）

カメラのレンズは、ピントを合わせた距離だけがくっきり写り、そこから前や後ろに離れるほどボケます。くっきり写る距離の範囲を**被写界深度**と呼び、絞りを開くほど狭く（ボケが大きく）なります。スマートフォンのポートレートモードは、レンズの代わりに、推定した奥行きを使ってこのボケを計算で作っています。

ここでは、ピントを合わせた点との**近さの差**が大きい画素ほど、大きな円形のぼかしをかけます。レンズの式から、ボケの大きさは「距離の逆数の差」に比例することが分かっています。Depth Anything V2 の相対的な奥行きは距離の逆数に近い量なので、その差をそのまま使えます。

- `FOCUS_X`、`FOCUS_Y`：ピントを合わせる位置（左上が 0、右下が 1。赤い十字）
- `STRENGTH`：ボケの強さ（ぼかしの半径の最大値、画素）。レンズの絞りを開くことにあたる

In [ ]:
IMAGE = "astronaut"  # @param ["astronaut", "motorcycle", "rocket", "coffee"] {allow-input: true}
FOCUS_X = 0.37  # @param {type:"slider", min:0, max:1, step:0.01}
FOCUS_Y = 0.25  # @param {type:"slider", min:0, max:1, step:0.01}
STRENGTH = 20  # @param {type:"slider", min:0, max:40, step:1}

image, depth = images[IMAGE], depths[IMAGE]
col, row = round(FOCUS_X * (image.width - 1)), round(FOCUS_Y * (image.height - 1))
focus = normalize(depth)[row, col]  # ピントを合わせる点の近さ（0〜1）
result = depth_of_field(image, depth, focus, STRENGTH)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(image)
axes[0].set_title("input")
axes[1].imshow(np.abs(normalize(depth) - focus), cmap="viridis")
axes[1].set_title("distance from focus (bright = more blur)")
axes[2].imshow(result)
axes[2].set_title(f"depth of field (strength {STRENGTH})")
for ax in axes:
    ax.plot(col, row, "+", color="red", markersize=20, markeredgewidth=2)
    ax.axis("off")
plt.tight_layout()
plt.show()

ピントの位置を顔、ヘルメット、背景の旗と動かしてみましょう。物の輪郭のまわりに、にじみが出ることがあります。本物のレンズは、手前の物に隠れた背景からの光も集めますが、写真1枚には隠れた背景の情報が無いためです。

### 6-2. ピントを動かす（ピント送りの動画）

ピントを合わせる近さを、一番近い所から一番遠い所まで少しずつ動かして、動画（GIF）にします。映画などで使われる「ピント送り」の効果です。

In [ ]:
IMAGE = "motorcycle"  # @param ["astronaut", "motorcycle", "rocket", "coffee"] {allow-input: true}
STRENGTH = 12  # @param {type:"slider", min:2, max:40, step:1}

image, depth = images[IMAGE], depths[IMAGE]
small = fit_size(image, 400)  # 動画は小さくして、早く作る
small_depth = cv2.resize(depth, small.size)
stack = blur_stack(small, STRENGTH)  # ぼかした画像は、最初にまとめて作っておく
focuses = np.linspace(1, 0, 12)      # 一番近い所（1）→ 一番遠い所（0）
frames = [depth_of_field(small, small_depth, f, STRENGTH, stack=stack) for f in focuses]
show_gif(frames + frames[::-1])      # 行って戻る

### 6-3. 霧をかける

霧やかすみの中では、遠くの物ほど、光が空気中の細かい粒に散らされて弱まり、霧の色に近づきます。光の弱まり方は、距離に対して指数関数的です（$t = e^{-\beta d}$。$d$ は距離、$\beta$ は霧の濃さ）。

$$\text{霧の中の写真} = \text{元の写真} \times t + \text{霧の色} \times (1 - t)$$

ここでは、相対的な奥行きから作った「遠さ」（0：一番近い 〜 1：一番遠い）を、距離の代わりに使う簡易版です。この式を逆に使って、霧の写真から霧を取り除く研究（霧除去）もあります。

In [ ]:
IMAGE = "rocket"  # @param ["astronaut", "motorcycle", "rocket", "coffee"] {allow-input: true}
DENSITY = 3.0  # @param {type:"slider", min:0, max:8, step:0.5}

image, depth = images[IMAGE], depths[IMAGE]
far = 1 - normalize(depth)                      # 遠さ（0：近い 〜 1：遠い）
t = np.exp(-DENSITY * far)[..., None]           # 光が届く割合
fog_color = np.array([225, 228, 235], dtype=np.float32)
foggy = (np.asarray(image, dtype=np.float32) * t + fog_color * (1 - t)).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(image)
axes[0].set_title("input")
axes[1].imshow(foggy)
axes[1].set_title(f"fog (density {DENSITY})")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

### 6-4. 3次元で見る（点群）

メートル単位の奥行きがあれば、画素を3次元の点（**点群**）に戻せます。カメラの中心から各画素の方向へ、推定した距離だけ進んだ所に点を置きます。

画素の方向を決めるには、カメラの**画角**（写る範囲の角度）が必要です。分からないので、`FOV`（横の画角、度）で仮に決めます。スマートフォンの標準のカメラは、横に約70度です（`motorcycle` のカメラは約41度）。

図はマウスで回したり（ドラッグ）、拡大したり（ホイール）できます。横から見ると、奥行きの推定がどうなっているかがよく分かります。

In [ ]:
IMAGE = "motorcycle"  # @param ["astronaut", "motorcycle", "rocket", "coffee"] {allow-input: true}
FOV = 41  # @param {type:"slider", min:20, max:120, step:1}

import plotly.graph_objects as go

image = images[IMAGE]
meters = estimate_depth(image, METRIC_MODEL)  # 5-3 で選んだモデル（屋内用か屋外用）
h, w = meters.shape
focal = (w / 2) / np.tan(np.radians(FOV) / 2)  # 画角から決めた焦点距離（画素）
step = max(1, int(np.ceil(np.sqrt(h * w / 40000))))  # 点を4万個くらいに間引く
rows, cols = np.mgrid[0:h:step, 0:w:step]
z = meters[rows, cols]
x = (cols - w / 2) * z / focal  # 右方向（m）
y = (rows - h / 2) * z / focal  # 下方向（m）
colors = np.asarray(image)[rows, cols].reshape(-1, 3)

fig = go.Figure(go.Scatter3d(
    x=x.ravel(), y=z.ravel(), z=-y.ravel(), mode="markers",
    marker=dict(size=1.5, color=[f"rgb({r},{g},{b})" for r, g, b in colors])))
fig.update_layout(height=600, margin=dict(l=0, r=0, t=30, b=0), title=f"{IMAGE} (FOV {FOV} deg)",
                  scene=dict(aspectmode="data", xaxis_title="x [m]", yaxis_title="depth [m]", zaxis_title="up [m]",
                             camera=dict(eye=dict(x=0.0, y=-1.6, z=0.4))))
fig.show()

## 7. まとめ

目安：約2分

| 分かったこと | 節 |
| --- | --- |
| 単眼深度推定は、写真1枚から画素ごとに奥行きを出す「密な予測」。式は第1回と同じ $Y = F(X; \theta)$ で、$Y$ が画像の形になる | 2-1 |
| Depth Anything V2 は、14×14 画素のパッチをトークンにする ViT（DINOv2）と、トークンを画像の形に戻す DPT でできている | 2-3 |
| 正確な合成画像で先生役を学習し、先生が実写の写真に付けた仮の正解で生徒役を学習する | 2-4 |
| 基本のモデルは相対的な奥行き（単位なし）。メートル単位には、追加学習したモデルを使う | 2-2、5-3 |
| 入力を大きくすると細かくなるが、トークンが増えて計算が重くなる | 5-1 |
| 推定した奥行きは、ぼかし、霧、3次元の復元など、別の処理の材料になる | 6. |

**苦手なこと**（4. で見つけたものはありますか）

- 鏡やガラス：映り込んだ景色や、向こう側の景色の奥行きを出してしまうことがある
- 写真やポスター、画面を撮った写真：本当は平らでも、写っている場面の奥行きを出してしまう
- 相対的な奥行きは、写真ごとに目盛りが違う。2枚の写真の値を、そのまま比べることはできない

**発展**：同じモデルを、論文の著者が公開している GitHub のコードから動かす手順は、`02_run_github_code.ipynb` にあります。